# Tech LCA — Foreground database builder

Builds the `hydrogen foreground` Brightway database from the SimaPro-derived inventories
(SMR, SMR-CCS, CCS waste treatment, MP-E, AE construction + operation,
PEM construction + operation, SOEC construction + operation).

All settings come from [dashboard_config.py](dashboard_config.py).
Run this notebook when `RUN_BUILD_FOREGROUND_DATABASE = True`.

In [1]:
# All settings come from dashboard_config.py — the single master dashboard.
# Edit values there once; every notebook picks them up.
from dashboard_config import *
import dashboard_config as cfg
import lca_helpers as H

print_dashboard()
print()
ei, bio, fg_db, method = H.setup_brightway()

/opt/miniconda3/envs/brightway/lib/python3.11/site-packages/bw2calc/__init__.py:54: UserWarning: 
It seems like you have an ARM architecture, but haven't installed scikit-umfpack:

    https://pypi.org/project/scikit-umfpack/

Installing it could give you much faster calculations.

  warnings.warn(UMFPACK_WARNING)


Master Dashboard
----------------
Project:                 hydrogen-smr
Foreground DB:           hydrogen foreground
Run adaptive foreground: False
Run adaptive grid:       True
Run adaptive wind/grid:  True
Run adaptive prices:     False
Grid data source:        carbon_api | notebook: 4.1.custom_grid_carbon_intensity_api.ipynb
Run grid scenario LCA:   True
Run wind/grid LCA:       True
Run price data:          True
Grid method:             cheap | loss factor: 1.0316426921769999
Wind/grid method:        cheap
Selected grid techs:     ['Alkaline electrolyser, Hermesmann']
Wind/grid mode:          blended  | electrolyser(s): ['Alkaline electrolyser, Hermesmann']
Price output dir:        price_outputs
Elexon provider:         APXMIDP | fallback: N2EXMIDP
Cost case:               central | wind: central

Current Brightway project: hydrogen-smr
Using ecoinvent database: ecoinvent-3.9.1-apos
Using biosphere database: ecoinvent-3.9.1-biosphere
Using foreground database: hydrogen foreground
U

In [2]:
if not RUN_BUILD_FOREGROUND_DATABASE:
    raise SystemExit(
        "RUN_BUILD_FOREGROUND_DATABASE is False in dashboard_config.py. "
        "Set it True and re-run this notebook to (re)build the foreground database."
    )

## SMR

In [3]:
queries = {
    "electricity": "market for electricity high voltage GB",
    "gas":         "market for natural gas high pressure GB",
    "tap_water":   "market for tap water Europe without Switzerland",
    "concrete":    "market for concrete normal RoW",
    "steel":       "market for steel unalloyed GLO",
    "aluminium":   "aluminium primary cast alloy slab continuous casting GLO",
    "cast_iron":   "market for cast iron GLO",
    "gas_turbine": "gas turbine 10MW electrical GLO",
    "wastewater":  "market for wastewater average Europe without Switzerland",
}
candidate_index = {
    "electricity": 10, "gas": 1, "tap_water": 0, "concrete": 0, "steel": 0,
    "aluminium": 0, "cast_iron": 0, "gas_turbine": 0, "wastewater": 0,
}
activities = {}
for key, q in queries.items():
    print("\n" + "=" * 90)
    print(key, ":", q)
    activities[key] = H.pick_candidate(q, index=candidate_index[key], database=ei)


electricity : market for electricity high voltage GB
    0 | name: electricity, high voltage, import from GB | ref: electricity, high voltage | unit: kilowatt hour | loc: BE
    1 | name: electricity, high voltage, import from GB | ref: electricity, high voltage | unit: kilowatt hour | loc: NL
    2 | name: electricity, high voltage, import from GB | ref: electricity, high voltage | unit: kilowatt hour | loc: FR
    3 | name: electricity, high voltage, import from GB | ref: electricity, high voltage | unit: kilowatt hour | loc: IE
    4 | name: electricity, high voltage, import from IE | ref: electricity, high voltage | unit: kilowatt hour | loc: GB
    5 | name: electricity, high voltage, import from NL | ref: electricity, high voltage | unit: kilowatt hour | loc: GB
    6 | name: electricity, high voltage, import from FR | ref: electricity, high voltage | unit: kilowatt hour | loc: GB
    7 | name: electricity, high voltage, import from BE | ref: electricity, high voltage | unit: ki

## SMR-CCS

In [4]:
queries_ccs = {
    "electricity":  "market for electricity high voltage GB",
    "gas":          "market for natural gas high pressure GB",
    "tap_water":    "market for tap water Europe without Switzerland",
    "concrete":     "market for concrete normal RoW",
    "steel":        "market for steel unalloyed GLO",
    "aluminium":    "aluminium primary cast alloy slab continuous casting GLO",
    "cast_iron":    "market for cast iron GLO",
    "gas_turbine":  "gas turbine 10MW electrical GLO",
    "wastewater":   "market for wastewater average Europe without Switzerland",
    "ccs_pipeline": "pipeline natural gas long distance low capacity onshore GLO",
    "ccs_well":     "onshore well oil gas GLO",
    "ccs_diesel":   "market group for diesel RER",
}
candidate_index_ccs = {
    "electricity": 10, "gas": 1, "tap_water": 0, "concrete": 0, "steel": 0,
    "aluminium": 0, "cast_iron": 0, "gas_turbine": 0, "wastewater": 0,
    "ccs_pipeline": 0, "ccs_well": 1, "ccs_diesel": 0,
}
activities_ccs = {}
for key, q in queries_ccs.items():
    print("\n" + "=" * 90)
    print(key, ":", q)
    activities_ccs[key] = H.pick_candidate(q, index=candidate_index_ccs[key], database=ei)


electricity : market for electricity high voltage GB
    0 | name: electricity, high voltage, import from GB | ref: electricity, high voltage | unit: kilowatt hour | loc: BE
    1 | name: electricity, high voltage, import from GB | ref: electricity, high voltage | unit: kilowatt hour | loc: NL
    2 | name: electricity, high voltage, import from GB | ref: electricity, high voltage | unit: kilowatt hour | loc: FR
    3 | name: electricity, high voltage, import from GB | ref: electricity, high voltage | unit: kilowatt hour | loc: IE
    4 | name: electricity, high voltage, import from IE | ref: electricity, high voltage | unit: kilowatt hour | loc: GB
    5 | name: electricity, high voltage, import from NL | ref: electricity, high voltage | unit: kilowatt hour | loc: GB
    6 | name: electricity, high voltage, import from FR | ref: electricity, high voltage | unit: kilowatt hour | loc: GB
    7 | name: electricity, high voltage, import from BE | ref: electricity, high voltage | unit: ki

## Methane Pyrolysis (MP-E)

In [5]:
queries_MP = {
    "electricity":        "market for electricity high voltage GB",
    "gas":                "market for natural gas high pressure GB",
    "high_alloyed_steel": "Market for steel, chromium steel 18/8 GLO",
    "low_alloyed_steel":  "Market for steel, low-alloyed GLO",
    "palladium":          "Market for palladium GLO",
    "copper":             "Cathode Market for copper GLO",
    "silica_sand":        "Market for silica sand GLO",
    "tin":                "Market for tin GLO",
    "silicon_carbide":    "Market for silicon carbide GLO",
}
candidate_index_MP = {
    "electricity": 10, "gas": 1,
    "high_alloyed_steel": 1, "low_alloyed_steel": 1, "palladium": 1,
    "copper": 0, "silica_sand": 0, "tin": 0, "silicon_carbide": 0,
}
activities_MP = {}
for key, q in queries_MP.items():
    print("\n" + "=" * 90)
    print(key, ":", q)
    activities_MP[key] = H.pick_candidate(q, index=candidate_index_MP[key], database=ei)


electricity : market for electricity high voltage GB
    0 | name: electricity, high voltage, import from GB | ref: electricity, high voltage | unit: kilowatt hour | loc: BE
    1 | name: electricity, high voltage, import from GB | ref: electricity, high voltage | unit: kilowatt hour | loc: NL
    2 | name: electricity, high voltage, import from GB | ref: electricity, high voltage | unit: kilowatt hour | loc: FR
    3 | name: electricity, high voltage, import from GB | ref: electricity, high voltage | unit: kilowatt hour | loc: IE
    4 | name: electricity, high voltage, import from IE | ref: electricity, high voltage | unit: kilowatt hour | loc: GB
    5 | name: electricity, high voltage, import from NL | ref: electricity, high voltage | unit: kilowatt hour | loc: GB
    6 | name: electricity, high voltage, import from FR | ref: electricity, high voltage | unit: kilowatt hour | loc: GB
    7 | name: electricity, high voltage, import from BE | ref: electricity, high voltage | unit: ki

## Alkaline Electrolyser (capital good)

In [6]:
queries_AE = {
    "polyethylene_hd":         "polyethylene production high density granulate",
    "extrusion_pipes":         "extrusion plastic pipes market",
    "reinforcing_steel":       "reinforcing steel production",
    "sheet_rolling_steel":     "sheet rolling steel GLO market",
    "electronics":             "electronics production control units",
    "aluminium_wrought":       "aluminium wrought alloy GLO market",
    "copper":                  "market for copper GLO",
    "sheet_rolling_aluminium": "sheet rolling aluminium GLO market",
    "tube_insulation":         "tube insulation elastomere",
    "wire_drawing_copper":     "wire drawing copper GLO market",
    "glass_fibre":             "glass fibre production",
    "sheet_rolling_cr_steel":  "sheet rolling chromium steel GLO market",
    "cr_steel_hot_rolled":     "steel chromium steel 18/8 hot rolled production",
    "cast_iron_ae":            "cast iron production",
    "ethylene_glycol":         "ethylene glycol production",
    "welding_arc_steel":       "welding arc steel GLO market",
    "polypropylene":           "polypropylene granulate production",
    "injection_moulding":      "injection moulding GLO market",
    "low_alloyed_steel_hr":    "steel low-alloyed hot rolled production",
    "concrete_35mpa":          "concrete production 35MPa",
    "nickel":                  "Nickel, class 1 market for nickel, class 1 ",
    "tetrafluoroethylene":     "tetrafluoroethylene production",
    "polysulfone":             "polysulfone production membrane filtration",
    "zirconium_oxide":         "zirconium oxide production",
    "electricity_lv":          "market for electricity low voltage DE",
}
candidate_index_AE = {
    "polyethylene_hd": 8, "extrusion_pipes": 0, "reinforcing_steel": 2,
    "sheet_rolling_steel": 0, "electronics": 1, "aluminium_wrought": 1,
    "copper": 2, "sheet_rolling_aluminium": 0, "tube_insulation": 1,
    "wire_drawing_copper": 0, "glass_fibre": 2, "sheet_rolling_cr_steel": 0,
    "cr_steel_hot_rolled": 1, "cast_iron_ae": 6, "ethylene_glycol": 0,
    "welding_arc_steel": 0, "polypropylene": 1, "injection_moulding": 0,
    "low_alloyed_steel_hr": 1, "concrete_35mpa": 0, "nickel": 0,
    "tetrafluoroethylene": 2, "polysulfone": 0, "zirconium_oxide": 1,
    "electricity_lv": 0,
}
activities_AE = {}
for key, q in queries_AE.items():
    print("\n" + "=" * 90)
    print(key, ":", q)
    activities_AE[key] = H.pick_candidate(q, index=candidate_index_AE[key], database=ei)


polyethylene_hd : polyethylene production high density granulate
    0 | name: polyethylene, high density, granulate, recycled to generic market for high density PE granulate | ref: polyethylene, high density, granulate | unit: kilogram | loc: RoW
    1 | name: polyethylene production, high density, granulate, recycled | ref: polyethylene, high density, granulate, recycled | unit: kilogram | loc: RoW
    2 | name: polyethylene production, high density, granulate, recycled | ref: polyethylene, high density, granulate, recycled | unit: kilogram | loc: CH
    3 | name: polyethylene production, high density, granulate, recycled | ref: polyethylene, high density, granulate, recycled | unit: kilogram | loc: US
    4 | name: polyethylene production, high density, granulate, recycled | ref: polyethylene, high density, granulate, recycled | unit: kilogram | loc: Europe without Switzerland
    5 | name: polyethylene production, low density, granulate | ref: polyethylene, low density, granulate 

## Alkaline operation

In [7]:
queries_AE_op = {
    "water_deionised": "market for water deionised Europe without Switzerland",
    "water_softened":  "market for water completely softened RER",
    "electrolyte_koh": "market for electrolyte KOH LiOH additive",
    "electricity_lv":  "market for electricity low voltage DE",
}
candidate_index_AE_op = {"water_deionised": 0, "water_softened": 0,
                         "electrolyte_koh": 0, "electricity_lv": 0}
activities_AE_op = {}
for key, q in queries_AE_op.items():
    print("\n" + "=" * 90)
    print(key, ":", q)
    activities_AE_op[key] = H.pick_candidate(q, index=candidate_index_AE_op[key], database=ei)


water_deionised : market for water deionised Europe without Switzerland
    0 | name: market for water, deionised | ref: water, deionised | unit: kilogram | loc: Europe without Switzerland

  → Selected [0]: market for water, deionised | Europe without Switzerland

water_softened : market for water completely softened RER
    0 | name: market for water, completely softened | ref: water, completely softened | unit: kilogram | loc: RER

  → Selected [0]: market for water, completely softened | RER

electrolyte_koh : market for electrolyte KOH LiOH additive
    0 | name: market for electrolyte, KOH, LiOH additive | ref: electrolyte, KOH, LiOH additive | unit: kilogram | loc: GLO

  → Selected [0]: market for electrolyte, KOH, LiOH additive | GLO

electricity_lv : market for electricity low voltage DE
    0 | name: market for electricity, low voltage | ref: electricity, low voltage | unit: kilowatt hour | loc: DE
    1 | name: electricity, low voltage, photovoltaic, import from Germany | 

## PEM Electrolyser construction

In [8]:
queries_PEM_con = {
    "aluminium_wrought":       "aluminium wrought alloy GLO market",
    "carbon_black":            "carbon black production GLO",
    "cast_iron_rer":           "cast iron production RER",
    "concrete_35mpa":          "concrete production 35MPa",
    "copper_cathode":          "copper cathode GLO market",
    "electronics_rer":         "electronics production control units RER",
    "ethylene_glycol_rer":     "ethylene glycol production RER",
    "extrusion_pipes":         "extrusion plastic pipes market",
    "injection_moulding":      "injection moulding GLO market",
    "lubricating_oil":         "market for lubricating oil",
    "polyethylene_ld":         "polyethylene production low density granulate RER",
    "polypropylene_rer":       "polypropylene granulate production RER",
    "reinforcing_steel_eur":   "reinforcing steel production Europe without Austria",
    "sheet_rolling_aluminium": "sheet rolling aluminium GLO market",
    "sheet_rolling_cr_steel":  "sheet rolling chromium steel GLO market",
    "sheet_rolling_copper":    "sheet rolling copper GLO market",
    "sheet_rolling_steel":     "sheet rolling steel GLO market",
    "cr_steel_hr_rer":         "steel chromium steel 18/8 hot rolled production RER",
    "low_alloyed_steel_hr_rer":"steel low-alloyed hot rolled production RER",
    "synthetic_rubber":        "synthetic rubber production RER",
    "tetrafluoroethylene":     "tetrafluoroethylene production",
    "titanium":                "titanium production GLO",
    "tube_insulation":         "tube insulation elastomere",
    "welding_arc_steel":       "welding arc steel GLO market",
    "wire_drawing_copper":     "wire drawing copper GLO market",
    "zeolite":                 "zeolite powder production RER",
    "electricity_lv_gb":       "market for electricity low voltage GB",
    "platinum":                "market for platinum GLO",
}
candidate_index_PEM_con = {
    "aluminium_wrought": 1, "carbon_black": 0, "cast_iron_rer": 0,
    "concrete_35mpa": 2, "copper_cathode": 0, "electronics_rer": 0,
    "ethylene_glycol_rer": 4, "extrusion_pipes": 0, "injection_moulding": 0,
    "lubricating_oil": 1, "polyethylene_ld": 0, "polypropylene_rer": 0,
    "reinforcing_steel_eur": 0, "sheet_rolling_aluminium": 0,
    "sheet_rolling_cr_steel": 0, "sheet_rolling_copper": 0,
    "sheet_rolling_steel": 0, "cr_steel_hr_rer": 0, "low_alloyed_steel_hr_rer": 0,
    "synthetic_rubber": 0, "tetrafluoroethylene": 3, "titanium": 2,
    "tube_insulation": 1, "welding_arc_steel": 0, "wire_drawing_copper": 0,
    "zeolite": 0, "electricity_lv_gb": 0, "platinum": 0,
}
activities_PEM_con = {}
for key, q in queries_PEM_con.items():
    print("\n" + "=" * 90)
    print(key, ":", q)
    activities_PEM_con[key] = H.pick_candidate(q, index=candidate_index_PEM_con[key], database=ei)


aluminium_wrought : aluminium wrought alloy GLO market
    0 | name: aluminium ingot, primary, to aluminium, wrought alloy market | ref: aluminium, wrought alloy | unit: kilogram | loc: GLO
    1 | name: market for aluminium, wrought alloy | ref: aluminium, wrought alloy | unit: kilogram | loc: GLO

  → Selected [1]: market for aluminium, wrought alloy | GLO

carbon_black : carbon black production GLO
    0 | name: carbon black production | ref: carbon black | unit: kilogram | loc: GLO
    1 | name: toner production, black, powder | ref: toner, black, powder | unit: kilogram | loc: GLO
    2 | name: strontium carbonate production | ref: strontium carbonate | unit: kilogram | loc: GLO
    3 | name: strontium carbonate production | ref: sodium sulfide | unit: kilogram | loc: GLO
    4 | name: boron carbide production | ref: boron carbide | unit: kilogram | loc: GLO
    5 | name: charcoal production | ref: charcoal | unit: kilogram | loc: GLO
    6 | name: lignite mine operation | ref: l

## PEM Electrolyser operation

In [9]:
queries_PEM_op = {
    "water_deionised":   "market for water deionised Europe without Switzerland",
    "water_softened":    "market for water completely softened RER",
    "electricity_lv_gb": "market for electricity low voltage GB",
    "heat_rer":          "market group for heat district or industrial other than natural gas RER",
}
candidate_index_PEM_op = {"water_deionised": 0, "water_softened": 0,
                          "electricity_lv_gb": 0, "heat_rer": 0}
activities_PEM_op = {}
for key, q in queries_PEM_op.items():
    print("\n" + "=" * 90)
    print(key, ":", q)
    activities_PEM_op[key] = H.pick_candidate(q, index=candidate_index_PEM_op[key], database=ei)


water_deionised : market for water deionised Europe without Switzerland
    0 | name: market for water, deionised | ref: water, deionised | unit: kilogram | loc: Europe without Switzerland

  → Selected [0]: market for water, deionised | Europe without Switzerland

water_softened : market for water completely softened RER
    0 | name: market for water, completely softened | ref: water, completely softened | unit: kilogram | loc: RER

  → Selected [0]: market for water, completely softened | RER

electricity_lv_gb : market for electricity low voltage GB
    0 | name: market for electricity, low voltage | ref: electricity, low voltage | unit: kilowatt hour | loc: GB
    1 | name: market for electricity, medium voltage | ref: electricity, medium voltage | unit: kilowatt hour | loc: GB
    2 | name: electricity, low voltage, residual mix | ref: electricity, low voltage | unit: kilowatt hour | loc: GB
    3 | name: market for electricity, for reuse in municipal waste incineration only | r

## SOEC Electrolyser construction

In [10]:
queries_SOEC_con = {
    "aluminium_wrought":        "aluminium wrought alloy GLO market",
    "cast_iron_rer":            "cast iron production RER",
    "concrete_35mpa":           "concrete production 35MPa",
    "copper_cathode":           "copper cathode GLO market",
    "electronics_rer":          "electronics production control units RER",
    "ethylene_glycol_rer":      "ethylene glycol production RER",
    "extrusion_pipes":          "extrusion plastic pipes market",
    "injection_moulding":       "injection moulding GLO market",
    "polyethylene_ld":          "polyethylene production low density granulate RER",
    "reinforcing_steel_eur":    "reinforcing steel production Europe without Austria",
    "sheet_rolling_aluminium":  "sheet rolling aluminium GLO market",
    "sheet_rolling_cr_steel":   "sheet rolling chromium steel GLO market",
    "sheet_rolling_steel":      "sheet rolling steel GLO market",
    "cr_steel_hr_rer":          "steel chromium steel 18/8 hot rolled production RER",
    "low_alloyed_steel_hr_rer": "steel low-alloyed hot rolled production RER",
    "tube_insulation":          "tube insulation elastomere",
    "welding_arc_steel":        "welding arc steel GLO market",
    "wire_drawing_copper":      "wire drawing copper GLO market",
    "abs_row":                  "acrylonitrile-butadiene-styrene copolymer production RoW",
    "aluminium_oxide":          "aluminium oxide metallurgical IAI market",
    "barium_oxide":             "barium oxide production GLO",
    "boric_oxide":              "boric oxide production GLO",
    "cerium_oxide":             "cerium oxide market GLO",
    "nickel_class1":            "nickel class 1 market GLO",
    "praseodymium_oxide":       "praseodymium oxide market GLO",
    "samarium_seg_oxide":       "samarium europium gadolinium oxide market GLO",
    "silicone_product":         "silicone product production RER",
    "zirconium_oxide":          "zirconium oxide production RoW",
    "electricity_lv_gb":        "market for electricity low voltage GB",
}
candidate_index_SOEC_con = {
    "aluminium_wrought": 1, "cast_iron_rer": 0, "concrete_35mpa": 2,
    "copper_cathode": 0, "electronics_rer": 0, "ethylene_glycol_rer": 4,
    "extrusion_pipes": 0, "injection_moulding": 0, "polyethylene_ld": 0,
    "reinforcing_steel_eur": 0, "sheet_rolling_aluminium": 0,
    "sheet_rolling_cr_steel": 0, "sheet_rolling_steel": 0,
    "cr_steel_hr_rer": 0, "low_alloyed_steel_hr_rer": 0,
    "tube_insulation": 1, "welding_arc_steel": 0, "wire_drawing_copper": 0,
    "abs_row": 0, "aluminium_oxide": 0, "barium_oxide": 0, "boric_oxide": 0,
    "cerium_oxide": 0, "nickel_class1": 0, "praseodymium_oxide": 0,
    "samarium_seg_oxide": 0, "silicone_product": 0, "zirconium_oxide": 0,
    "electricity_lv_gb": 0,
}
activities_SOEC_con = {}
for key, q in queries_SOEC_con.items():
    print("\n" + "=" * 90)
    print(key, ":", q)
    activities_SOEC_con[key] = H.pick_candidate(q, index=candidate_index_SOEC_con[key], database=ei)


aluminium_wrought : aluminium wrought alloy GLO market
    0 | name: aluminium ingot, primary, to aluminium, wrought alloy market | ref: aluminium, wrought alloy | unit: kilogram | loc: GLO
    1 | name: market for aluminium, wrought alloy | ref: aluminium, wrought alloy | unit: kilogram | loc: GLO

  → Selected [1]: market for aluminium, wrought alloy | GLO

cast_iron_rer : cast iron production RER
    0 | name: cast iron production | ref: cast iron | unit: kilogram | loc: RER
    1 | name: pig iron production | ref: pig iron | unit: kilogram | loc: RER
    2 | name: steel production, converter, low-alloyed | ref: steel, low-alloyed | unit: kilogram | loc: RER
    3 | name: steel production, electric, chromium steel 18/8 | ref: steel, chromium steel 18/8 | unit: kilogram | loc: RER
    4 | name: steel production, converter, unalloyed | ref: steel, unalloyed | unit: kilogram | loc: RER
    5 | name: 2,4-dichlorophenol production | ref: 2,4-dichlorophenol | unit: kilogram | loc: RER
  

## SOEC Electrolyser operation

In [11]:
queries_SOEC_op = {
    "water_deionised":   "market for water deionised Europe without Switzerland",
    "water_softened":    "market for water completely softened RER",
    "electricity_lv_gb": "market for electricity low voltage GB",
    "heat_rer":          "market group for heat district or industrial other than natural gas RER",
}
candidate_index_SOEC_op = {"water_deionised": 0, "water_softened": 0,
                           "electricity_lv_gb": 0, "heat_rer": 0}
activities_SOEC_op = {}
for key, q in queries_SOEC_op.items():
    print("\n" + "=" * 90)
    print(key, ":", q)
    activities_SOEC_op[key] = H.pick_candidate(q, index=candidate_index_SOEC_op[key], database=ei)


water_deionised : market for water deionised Europe without Switzerland
    0 | name: market for water, deionised | ref: water, deionised | unit: kilogram | loc: Europe without Switzerland

  → Selected [0]: market for water, deionised | Europe without Switzerland

water_softened : market for water completely softened RER
    0 | name: market for water, completely softened | ref: water, completely softened | unit: kilogram | loc: RER

  → Selected [0]: market for water, completely softened | RER

electricity_lv_gb : market for electricity low voltage GB
    0 | name: market for electricity, low voltage | ref: electricity, low voltage | unit: kilowatt hour | loc: GB
    1 | name: market for electricity, medium voltage | ref: electricity, medium voltage | unit: kilowatt hour | loc: GB
    2 | name: electricity, low voltage, residual mix | ref: electricity, low voltage | unit: kilowatt hour | loc: GB
    3 | name: market for electricity, for reuse in municipal waste incineration only | r

## Match the direct biosphere flows

In [12]:
print("CO2 candidates")
co2_candidates = H.show_biosphere_candidates("Carbon dioxide fossil", max_results=10)
co2 = co2_candidates[0]
print("\nUsing CO2 flow:", co2)

optional_biosphere = {}
for key, q in {
    "oxygen_air":     "Oxygen",
    "water_air":      "Water air",
    "water_resource": "Water cooling unspecified natural origin",
}.items():
    print("\n" + "=" * 90)
    print(key, ":", q)
    candidates = H.show_biosphere_candidates(q, max_results=8)
    optional_biosphere[key] = candidates[0] if candidates else None
    if candidates:
        print("Default optional match:", candidates[0])
    else:
        print("No match found; this optional flow will be skipped.")

CO2 candidates
    0 | name: Carbon dioxide, fossil | unit: kilogram | categories: ('air',)
    1 | name: Carbon dioxide, non-fossil | unit: kilogram | categories: ('air',)
    2 | name: Carbon dioxide, fossil | unit: kilogram | categories: ('air', 'lower stratosphere + upper troposphere')
    3 | name: Carbon dioxide, non-fossil | unit: kilogram | categories: ('air', 'lower stratosphere + upper troposphere')
    4 | name: Carbon dioxide, fossil | unit: kilogram | categories: ('air', 'low population density, long-term')
    5 | name: Carbon dioxide, fossil | unit: kilogram | categories: ('air', 'urban air close to ground')
    6 | name: Carbon dioxide, non-fossil, resource correction | unit: kilogram | categories: ('natural resource', 'in air')
    7 | name: Carbon dioxide, non-fossil | unit: kilogram | categories: ('air', 'urban air close to ground')
    8 | name: Carbon dioxide, non-fossil | unit: kilogram | categories: ('air', 'low population density, long-term')
    9 | name: Carbo

## SMR inventory

In [13]:
technosphere_exchanges = [
    ("electricity", -1.1,    "kilowatt hour", "Avoided product: Electricity, high voltage GB"),
    ("gas",         4.85,    "cubic meter",   "Natural gas, high pressure GB"),
    ("tap_water",   6.64,    "kilogram",      "Tap water"),
    ("concrete",    6.60e-6, "cubic meter",   "Concrete, normal RoW"),
    ("steel",       5.06e-3, "kilogram",      "Steel, unalloyed GLO"),
    ("aluminium",   4.17e-5, "kilogram",      "Aluminium, primary, cast alloy slab"),
    ("cast_iron",   6.18e-5, "kilogram",      "Cast iron GLO"),
    ("gas_turbine", 7.90e-10,"unit",          "Gas turbine, 10MW electrical GLO"),
    ("wastewater",  0.00172, "cubic meter",   "Wastewater, average"),
]
biosphere_exchanges = [
    (co2, 9.0, "kilogram", "Carbon dioxide; modelled as fossil CO2"),
    (optional_biosphere.get("oxygen_air"),     0.31, "kilogram",    "Oxygen to air"),
    (optional_biosphere.get("water_air"),      2.93, "kilogram",    "Water to air"),
    (optional_biosphere.get("water_resource"), 0.38, "cubic meter", "Water, cooling, GB"),
]
biosphere_exchanges = [r for r in biosphere_exchanges if r[0] is not None]
print("SMR exchanges:", len(technosphere_exchanges), "techno /", len(biosphere_exchanges), "bio")

SMR exchanges: 9 techno / 4 bio


## SMR-CCS inventory

In [14]:
SMRCCS_CODE = "smr_ccs_hermesmann_1kg_h2"
CCS_CODE    = "ccs_hermesmann_1kg_co2"

technosphere_exchanges_ccs = [
    ("electricity", -0.05,    "kilowatt hour", "Avoided electricity"),
    ("gas",          5.33,    "cubic meter",   "Natural gas, high pressure GB"),
    ("tap_water",    4.68,    "kilogram",      "Tap water"),
    ("concrete",     6.60e-6, "cubic meter",   "Concrete RoW"),
    ("steel",        5.06e-3, "kilogram",      "Steel unalloyed GLO"),
    ("aluminium",    4.17e-5, "kilogram",      "Aluminium, primary"),
    ("cast_iron",    6.18e-5, "kilogram",      "Cast iron GLO"),
    ("gas_turbine",  7.90e-10,"unit",          "Gas turbine 10MW"),
    ("wastewater",   0.00183, "cubic meter",   "Wastewater, average"),
]
biosphere_exchanges_ccs = [
    (co2,                                      0.99, "kilogram",    "CO2 residual, not captured"),
    (optional_biosphere.get("oxygen_air"),     0.41, "kilogram",    "Oxygen to air"),
    (optional_biosphere.get("water_air"),      1.33, "kilogram",    "Water to air"),
    (optional_biosphere.get("water_resource"), 1.18, "cubic meter", "Water, cooling, GB"),
]
biosphere_exchanges_ccs = [r for r in biosphere_exchanges_ccs if r[0] is not None]
print("SMR-CCS exchanges:", len(technosphere_exchanges_ccs),
      "techno (+1 CCS foreground link) /", len(biosphere_exchanges_ccs), "bio")

SMR-CCS exchanges: 9 techno (+1 CCS foreground link) / 4 bio


## CCS waste-treatment inventory

In [15]:
technosphere_exchanges_ccs_wt = [
    ("ccs_pipeline", 2.34666e-9, "kilometer",     "Pipeline, natural gas, long distance"),
    ("ccs_well",     9.863e-8,   "meter",         "Onshore well, oil/gas"),
    ("ccs_diesel",   0.0154,     "kilogram",      "Diesel"),
    ("electricity",  0.0171,     "kilowatt hour", "Electricity HV GB — pipeline pumping"),
    ("electricity",  0.0070,     "kilowatt hour", "Electricity HV GB — compressor"),
]
biosphere_exchanges_ccs_wt = [
    (co2, 1.36106e-7, "kilogram", "CO2 fugitive emissions from CCS pipeline/well"),
]
biosphere_exchanges_ccs_wt = [r for r in biosphere_exchanges_ccs_wt if r[0] is not None]
print("CCS waste-treatment:", len(technosphere_exchanges_ccs_wt), "techno /",
      len(biosphere_exchanges_ccs_wt), "bio")

CCS waste-treatment: 5 techno / 1 bio


## MP-E inventory

In [16]:
MP_E_CODE = "mp_e_1kg_h2"
technosphere_exchanges_MP_E = [
    ("gas",               5.72,    "cubic meter",   "Natural gas, high pressure GB"),
    ("electricity",      10.29,    "kilowatt hour", "Electricity HV GB"),
    ("palladium",         8.45e-6, "kilogram",      "Palladium GLO"),
    ("copper",            5.63e-6, "kilogram",      "Copper GLO"),
    ("low_alloyed_steel", 1.99e-3, "kilogram",      "Steel low-alloyed"),
    ("high_alloyed_steel",3.89e-4, "kilogram",      "Steel chromium 18/8"),
    ("silica_sand",       6.18e-5, "kilogram",      "Silica sand"),
    ("tin",               2.58e-2, "kilogram",      "Tin"),
    ("silicon_carbide",   3.23e-6, "kilogram",      "Silicon carbide"),
]
biosphere_exchanges_MP_E = []
print("MP-E exchanges:", len(technosphere_exchanges_MP_E), "techno / 0 bio")

MP-E exchanges: 9 techno / 0 bio


## Alkaline Electrolyser inventory

In [17]:
AE_CODE = "ae_hermesmann_1unit"
technosphere_exchanges_AE = [
    ("polyethylene_hd",          464.6,    "kilogram", "Water purifier / feed tank"),
    ("extrusion_pipes",          464.6,    "kilogram", "Water purifier / feed tank"),
    ("reinforcing_steel",       5214.4,    "kilogram", "Reinforcing steel, aggregated BoP"),
    ("sheet_rolling_steel",    10215.0,    "kilogram", "Sheet rolling steel, aggregated BoP"),
    ("electronics",              100.0,    "kilogram", "Control panel / electronics"),
    ("aluminium_wrought",        160.0,    "kilogram", "Transformer + frequency converter"),
    ("copper",                   616.7,    "kilogram", "Transformer + compressor + tubing/cables"),
    ("sheet_rolling_aluminium",  100.0,    "kilogram", "Transformer and rectifier"),
    ("tube_insulation",          207.9,    "kilogram", "Transformer + compressor + tubing"),
    ("wire_drawing_copper",      616.7,    "kilogram", "Transformer + compressor + tubing"),
    ("glass_fibre",              464.6,    "kilogram", "H2 drier and deoxidiser"),
    ("sheet_rolling_cr_steel", 26892.2,    "kilogram", "Sheet rolling Cr steel, aggregated"),
    ("cr_steel_hot_rolled",    26892.2,    "kilogram", "Cr steel 18/8 hot rolled, aggregated"),
    ("cast_iron_ae",             716.1,    "kilogram", "Compressor + pumps and coolers"),
    ("ethylene_glycol",            7.0,    "kilogram", "Diaphragm for compressor"),
    ("welding_arc_steel",         29.0,    "meter",    "Buffertank"),
    ("polypropylene",              3.0,    "kilogram", "Alkali-resistant rotary pump"),
    ("injection_moulding",         3.0,    "kilogram", "Alkali-resistant rotary pump"),
    ("low_alloyed_steel_hr",    6075.6,    "kilogram", "Container"),
    ("concrete_35mpa",             7.7,    "cubic meter", "Fundament"),
    ("nickel",                  2884.9,    "kilogram", "Anode + cathode with frame"),
    ("tetrafluoroethylene",      144.2,    "kilogram", "Gasket"),
    ("polysulfone",               48.8,    "kilogram", "Diaphragm zirfon"),
    ("zirconium_oxide",           73.0,    "kilogram", "Diaphragm zirfon"),
    ("electricity_lv",        323773.4,    "kilowatt hour", "BoP + Stack manufacturing electricity"),
]
biosphere_exchanges_AE = []
print("AE exchanges:", len(technosphere_exchanges_AE), "techno / 0 bio")

AE exchanges: 25 techno / 0 bio


## Alkaline Electrolysis operation inventory

In [18]:
AE_OP_CODE = "ae_op_hermesmann_1kg_h2"
technosphere_exchanges_AE_op = [
    ("water_deionised",  8.99,   "kilogram",      "Water, deionised"),
    ("water_softened",  88.1,    "kilogram",      "Water, completely softened"),
    ("electrolyte_koh",  0.0037, "kilogram",      "Electrolyte KOH/LiOH"),
    ("electricity_lv",  51.8,    "kilowatt hour", "Electricity LV GB — electrolysis"),
]
biosphere_exchanges_AE_op = []
print("AE op exchanges:", len(technosphere_exchanges_AE_op),
      "techno (+1 AE capital good link) / 0 bio")

AE op exchanges: 4 techno (+1 AE capital good link) / 0 bio


## PEM construction inventory

In [19]:
PEM_CON_CODE = "pem_con_hermesmann_1unit"
technosphere_exchanges_PEM_con = [
    ("extrusion_pipes",           464.6,    "kilogram",      "Water purifier / feed tank"),
    ("polyethylene_ld",           464.6,    "kilogram",      "Water purifier / feed tank"),
    ("aluminium_wrought",         287.0,    "kilogram",      "Power electronics + purification + end plate"),
    ("sheet_rolling_aluminium",   227.0,    "kilogram",      "Power electronics + purification + end plate"),
    ("electronics_rer",           100.0,    "kilogram",      "Control panel/electronics"),
    ("cast_iron_rer",             600.0,    "kilogram",      "Diaphragm compressor"),
    ("ethylene_glycol_rer",         7.0,    "kilogram",      "Diaphragm compressor"),
    ("copper_cathode",            349.5,    "kilogram",      "Power elec + water gas sep + compressor + current collector"),
    ("sheet_rolling_copper",      104.5,    "kilogram",      "Water gas separator + current collector"),
    ("wire_drawing_copper",       245.0,    "kilogram",      "Power electronics + compressor"),
    ("injection_moulding",        300.0,    "kilogram",      "Valve"),
    ("polypropylene_rer",         300.0,    "kilogram",      "Valve"),
    ("lubricating_oil",           100.0,    "kilogram",      "Back pressure regulator"),
    ("reinforcing_steel_eur",    3312.3,    "kilogram",      "Power elec + water purifier + compressor + container"),
    ("sheet_rolling_cr_steel",   4427.0,    "kilogram",      "Steel + heat ex + compressor + buffer + bolts"),
    ("sheet_rolling_steel",      5382.3,    "kilogram",      "Power elec + tubing + water purifier + compressor"),
    ("cr_steel_hr_rer",          4427.0,    "kilogram",      "Steel + heat ex + compressor + buffer + bolts (RER)"),
    ("low_alloyed_steel_hr_rer", 3150.0,    "kilogram",      "Tubing/pump + container"),
    ("welding_arc_steel",          29.0,    "meter",         "Buffer tank"),
    ("tube_insulation",           115.0,    "kilogram",      "Power electronics + compressor"),
    ("zeolite",                   100.0,    "kilogram",      "Ion exchanger"),
    ("concrete_35mpa",              2.3,    "cubic meter",   "Foundation"),
    ("carbon_black",                9.0,    "kilogram",      "Electrocatalyst anode + cathode"),
    ("platinum",                    0.875,  "kilogram",      "Electrocatalyst anode + cathode"),
    ("tetrafluoroethylene",        16.0,    "kilogram",      "Membrane polymer"),
    ("synthetic_rubber",            4.8,    "kilogram",      "Gasket"),
    ("titanium",                  528.0,    "kilogram",      "Bipolar plate"),
    ("electricity_lv_gb",      361672.3,    "kilowatt hour", "PEM stack + BoP manufacturing"),
]
biosphere_exchanges_PEM_con = []
print("PEM con exchanges:", len(technosphere_exchanges_PEM_con), "techno / 0 bio")

PEM con exchanges: 28 techno / 0 bio


## PEM operation inventory

In [20]:
PEM_OP_CODE = "pem_op_hermesmann_1kg_h2"
technosphere_exchanges_PEM_op = [
    ("water_deionised",    8.99,  "kilogram",      "Water, deionised"),
    ("water_softened",    88.1,   "kilogram",      "Water, completely softened"),
    ("electricity_lv_gb", 54.0,   "kilowatt hour", "Electricity LV GB"),
    ("heat_rer",           1.008, "megajoule",     "Heat district/industrial RER (0.28 kWh × 3.6)"),
]
biosphere_exchanges_PEM_op = []
print("PEM op exchanges:", len(technosphere_exchanges_PEM_op),
      "techno (+1 PEM capital good link) / 0 bio")

PEM op exchanges: 4 techno (+1 PEM capital good link) / 0 bio


## SOEC construction inventory

In [21]:
SOEC_CON_CODE = "soec_con_hermesmann_1unit"
technosphere_exchanges_SOEC_con = [
    ("extrusion_pipes",           534.0,    "kilogram",      "Water purifier / pre-heating"),
    ("polyethylene_ld",           534.0,    "kilogram",      "Water purifier / pre-heating"),
    ("aluminium_wrought",         401.0,    "kilogram",      "Freq converters + power electronics + water pump"),
    ("sheet_rolling_aluminium",   100.0,    "kilogram",      "Power electronics"),
    ("electronics_rer",           100.0,    "kilogram",      "Control electronics"),
    ("abs_row",                     1.4,    "kilogram",      "Control panel / heater"),
    ("injection_moulding",          1.4,    "kilogram",      "Control panel / heater"),
    ("cast_iron_rer",            3000.0,    "kilogram",      "Diaphragm compressors 1-5"),
    ("ethylene_glycol_rer",        35.0,    "kilogram",      "Diaphragm compressors 1-5"),
    ("copper_cathode",            428.5,    "kilogram",      "Control + freq conv + power electronics + freq conv water pump"),
    ("wire_drawing_copper",       428.5,    "kilogram",      "Control + freq conv + power electronics + freq conv water pump"),
    ("tube_insulation",           176.6,    "kilogram",      "Freq conv + power electronics + control + freq conv water pump"),
    ("reinforcing_steel_eur",   13730.6,    "kilogram",      "BoP steel components"),
    ("sheet_rolling_cr_steel",  25597.5,    "kilogram",      "BoP Cr steel components + air electrode"),
    ("sheet_rolling_steel",     12081.2,    "kilogram",      "BoP steel components"),
    ("cr_steel_hr_rer",         25597.5,    "kilogram",      "BoP Cr steel components + air electrode"),
    ("low_alloyed_steel_hr_rer", 3753.6,    "kilogram",      "Tubing + container — pre-heating"),
    ("welding_arc_steel",          33.3,    "meter",         "Buffer tank (pre-heating)"),
    ("concrete_35mpa",              2.3,    "cubic meter",   "Foundation"),
    ("aluminium_oxide",             6.4,    "kilogram",      "Electrolyte + H2 electrode"),
    ("barium_oxide",                6.4,    "kilogram",      "Electrolyte + H2 electrode"),
    ("boric_oxide",                 6.4,    "kilogram",      "Electrolyte + H2 electrode"),
    ("silicone_product",            6.4,    "kilogram",      "Electrolyte sealant"),
    ("nickel_class1",             144.1,    "kilogram",      "Air electrode + firing"),
    ("praseodymium_oxide",          9.0,    "kilogram",      "Air electrode screen printing"),
    ("samarium_seg_oxide",         37.7,    "kilogram",      "Blocking layer + screen printing"),
    ("cerium_oxide",               91.5,    "kilogram",      "Blocking layer screen printing"),
    ("zirconium_oxide",           170.7,    "kilogram",      "Blocking layer"),
    ("electricity_lv_gb",      443093.5,    "kilowatt hour", "SOEC BoP + Stack manufacturing"),
]
biosphere_exchanges_SOEC_con = []
print("SOEC con exchanges:", len(technosphere_exchanges_SOEC_con), "techno / 0 bio")

SOEC con exchanges: 29 techno / 0 bio


## SOEC operation inventory

In [22]:
SOEC_OP_CODE = "soec_op_hermesmann_1kg_h2"
technosphere_exchanges_SOEC_op = [
    ("water_deionised",    8.99,  "kilogram",      "Water, deionised"),
    ("water_softened",   644.7,   "kilogram",      "Water, completely softened"),
    ("electricity_lv_gb", 42.3,   "kilowatt hour", "Electricity LV GB"),
    ("heat_rer",          18.864, "megajoule",     "Heat district/industrial RER (5.24 kWh × 3.6)"),
]
biosphere_exchanges_SOEC_op = []
print("SOEC op exchanges:", len(technosphere_exchanges_SOEC_op),
      "techno (+1 SOEC capital good link) / 0 bio")

SOEC op exchanges: 4 techno (+1 SOEC capital good link) / 0 bio


## Data centre facility — Zhang et al. 2025 (Virginia hyperscale)

Cradle-to-grave inventory for a representative hyperscale data centre
(10,000 m² white space, Virginia US, SERC grid, PUE 1.4 baseline, 25-year
operating life), from Zhang, M., Carbajales-Dale, M., Ma, X., Guo, L., Fan, C.
(2025). "Cleaner grid or smarter cooling? Environmental impact trade-offs of a
data center using the life cycle assessment method." *Cleaner Energy Systems*
12, 100223. https://doi.org/10.1016/j.cles.2025.100223 — quantities are from
the paper's Table 1 / supplementary Table S1 (`...-mmc1.xlsx`); search queries
below are seeded from the exact ecoinvent process names the paper cites in its
own OpenLCA contribution breakdown (`...-mmc2.xlsx`), then re-verified against
this project's actual ecoinvent 3.9.1 apos database (see "Validated" below).
Functional unit is the whole facility over its 25-year operating life (`unit`
= 1), not a per-kWh or per-m² unit.

Unlike the hydrogen techs above, this is written to its **own** foreground
database (`DC_FOREGROUND_DB` in dashboard_config.py, not `FOREGROUND_DB`), so
rebuilding it never deletes/touches the hydrogen foreground and vice versa.
It's automatically selectable anywhere in this repo that calls
`H.list_foreground_databases()` / `H.list_process_activities()` — e.g. the
Setup LCA page's foreground picker — once written.

**Validated:** every `candidate_index` below was checked against a live
search of `ecoinvent-3.9.1-apos` (not just guessed) — all 27 queries resolve
to the intended activity. Running the LCA on the resulting activity with this
project's IPCC 2021 GWP100 method gives **8.38×10⁸ kg CO2-eq**, vs. the
paper's own reported baseline of 9.63×10⁸ kg CO2-eq (ratio 0.87 — a good match
given the different ecoinvent version/system model and LCIA method to the
paper's ecoinvent 3.7 cutoff + TRACI 2.1). The per-stage split also reproduces
the paper's own shape closely: operation ~99.99% of GWP (paper: 99.61%),
recycling ~−0.48% (paper: −0.52%), everything else sub-0.5% in both — so the
recycling-credit exchanges do **not** look double-counted against this apos
database. If you rebuild this later (e.g. after an ecoinvent update), it's
worth re-running that comparison rather than assuming the indices still pick
the same activities.

**Three deliberate substitutions** (flagged inline as comments in the queries
below) because ecoinvent 3.9.1 doesn't have exact equivalents of what the
paper's ecoinvent 3.7 cutoff model used:
- `battery` → a specific chemistry (NMC111) standing in for the old
  undifferentiated "Li-ion, rechargeable, prismatic" market, which 3.9.1 has
  split by chemistry (LiMn2O4 / NMC111 / NCA / NMC811 / LFP). Swap it for
  `LFP` if you know the facility's real UPS chemistry — LFP is more typical
  for modern stationary storage than any of these.
- `refrigerant_r134a` → the paper's "Tetrafluoroethane" flow is the same
  substance as R134a (1,1,1,2-tetrafluoroethane); 3.9.1 only has it under the
  refrigerant name.
- `servers` → ecoinvent's only embodied-manufacturing dataset for a generic
  server is `market for computer, laptop` — the same proxy the paper itself
  used (its own quantity is a server count with a refresh-cycle adjustment,
  not an actual laptop count).


In [23]:
queries_DC = {
    "concrete":                    "market for concrete 30-32MPa",
    "reinforcing_steel":           "market for reinforcing steel GLO",
    "structural_steel":            "market for steel low-alloyed hot rolled GLO",
    "copper":                      "market for copper cathode GLO",
    "aluminium_alloy":             "market for aluminium alloy metal matrix composite GLO",
    "insulation_pur":              "market for polyurethane rigid foam RoW",
    "transport_sea":               "market for transport freight sea container ship GLO",
    "transport_lorry_32t":         "market for transport freight lorry 16-32 metric ton EURO6 RoW",
    "transport_lorry_unspecified": "market for transport freight lorry unspecified RoW",
    "transport_train":             "market for transport freight train US",
    "diesel_building_machine":     "market for diesel burned in building machine GLO",
    "electricity_mv_serc":         "market for electricity medium voltage US-SERC",
    "tap_water":                   "market group for tap water GLO",
    "diesel_generator":            "market for diesel burned in diesel-electric generating set 10MW GLO",
    "servers":                     "market for computer laptop",  # ranks the plain "market for computer, laptop" at index 6, past several "operation, ..." usage-phase variants
    "battery":                     "market for battery Li-ion NMC111 rechargeable prismatic GLO",  # ecoinvent 3.9.1 split the old generic Li-ion market by chemistry; NMC111 picked as a representative default — reconsider if the real UPS chemistry is known (e.g. LFP is now common for stationary storage)
    "sodium_hypochlorite":         "market for sodium hypochlorite without water in 15% solution state RoW",
    "refrigerant_r134a":           "market for refrigerant R134a GLO",  # "tetrafluoroethane" in the source paper is the refrigerant R134a (1,1,1,2-tetrafluoroethane); ecoinvent 3.9.1 has no bare "tetrafluoroethane" market
    "scrap_aluminium":             "market for scrap aluminium RoW",
    "scrap_copper":                "market for scrap copper RoW",
    "scrap_steel":                 "market for scrap steel RoW",
    "weee_treatment":              "treatment of waste electric and electronic equipment shredding GLO",
    "waste_polyurethane_treatment": "treatment of waste polyurethane municipal incineration RoW",
    "waste_reinforced_concrete_treatment": "treatment of waste reinforced concrete recycling RoW",
    "avoided_aluminium_hydroxide": "aluminium hydroxide production RoW",
    "avoided_metal_working":       "metal working average for copper product manufacturing RoW",
    "avoided_steel":               "steel production converter low-alloyed RoW",
}
candidate_index_DC = {
    "concrete": 0, "reinforcing_steel": 0, "structural_steel": 0, "copper": 0,
    "aluminium_alloy": 0, "insulation_pur": 0, "transport_sea": 2,
    "transport_lorry_32t": 0, "transport_lorry_unspecified": 4, "transport_train": 0,
    "diesel_building_machine": 0, "electricity_mv_serc": 0, "tap_water": 0,
    "diesel_generator": 0, "servers": 6, "battery": 0, "sodium_hypochlorite": 0,
    "refrigerant_r134a": 1, "scrap_aluminium": 0, "scrap_copper": 0, "scrap_steel": 0,
    "weee_treatment": 0, "waste_polyurethane_treatment": 0,
    "waste_reinforced_concrete_treatment": 1, "avoided_aluminium_hydroxide": 0,
    "avoided_metal_working": 0, "avoided_steel": 0,
}
activities_DC = {}
for key, q in queries_DC.items():
    print("\n" + "=" * 90)
    print(key, ":", q)
    activities_DC[key] = H.pick_candidate(q, index=candidate_index_DC[key], database=ei)



concrete : market for concrete 30-32MPa
    0 | name: market for concrete, 30-32MPa | ref: concrete, 30-32MPa | unit: cubic meter | loc: RNA

  → Selected [0]: market for concrete, 30-32MPa | RNA

reinforcing_steel : market for reinforcing steel GLO
    0 | name: market for reinforcing steel | ref: reinforcing steel | unit: kilogram | loc: GLO
    1 | name: market for airport | ref: airport | unit: unit | loc: GLO
    2 | name: wind turbine construction, 4.5MW, onshore | ref: iron scrap, unsorted | unit: kilogram | loc: GLO
    3 | name: wind turbine construction, 4.5MW, onshore | ref: wind turbine, 4.5MW, onshore | unit: unit | loc: GLO

  → Selected [0]: market for reinforcing steel | GLO

structural_steel : market for steel low-alloyed hot rolled GLO
    0 | name: market for steel, low-alloyed, hot rolled | ref: steel, low-alloyed, hot rolled | unit: kilogram | loc: GLO
    1 | name: market for sheet rolling, steel | ref: sheet rolling, steel | unit: kilogram | loc: GLO

  → Select

## Data centre facility inventory

In [24]:
DC_CODE = "dc_zhang_virginia_baseline_25y"

technosphere_exchanges_DC = [
    # --- Material production ---
    ("concrete",          4000,   "cubic meter",   "Concrete, 30-32MPa, building shell and foundation"),
    ("reinforcing_steel",  4.80e5, "kilogram",      "Reinforcing steel"),
    ("structural_steel",   1.20e5, "kilogram",      "Structural steel"),
    ("copper",             2.00e4, "kilogram",      "Copper, cabling and architectural elements"),
    ("aluminium_alloy",    1.50e4, "kilogram",      "Aluminium alloy"),
    ("insulation_pur",     2.50e4, "kilogram",      "Insulation, rigid PUR foam"),

    # --- Transportation (materials to site) ---
    ("transport_sea",               3.036e7, "ton kilometer", "Freight, sea, container ship — Asia to US"),
    ("transport_lorry_32t",         6.10e5,  "ton kilometer", "Freight, lorry 16-32t EURO6 — regional trucking"),
    ("transport_lorry_unspecified", 9.00e4,  "ton kilometer", "Freight, lorry unspecified — regional trucking"),
    ("transport_train",             2.40e5,  "ton kilometer", "Freight train"),

    # --- Construction ---
    ("diesel_building_machine", 2.10e6, "megajoule",     "Diesel, burned in building machine — construction"),
    ("electricity_mv_serc",     7.8e4,  "kilowatt hour", "Electricity MV, US-SERC — construction (2.80e5 MJ / 3.6)"),
    ("tap_water",                1.2e6, "kilogram",      "Tap water, municipal — construction"),

    # --- Operation (25 years, PUE 1.4 baseline) ---
    ("electricity_mv_serc", 1.60e9, "kilowatt hour", "Electricity MV, US-SERC — IT + cooling/aux over 25y (5.76e9 MJ / 3.6)"),
    ("tap_water",            2.9e9, "kilogram",      "Tap water — cooling, WUE 1.8 L/kWh over 25y"),
    ("diesel_generator",    8.60e6, "megajoule",     "Diesel, burned in diesel-electric generating set 10MW — backup gen"),
    ("servers",              1.0e5, "unit",          "Computer/laptop (ecoinvent proxy for IT servers, incl. refresh cycles)"),
    ("battery",              2.0e4, "kilogram",      "Battery, Li-ion, rechargeable, prismatic — UPS"),
    ("sodium_hypochlorite", 1.30e5, "kilogram",      "Sodium hypochlorite — cooling water treatment"),
    ("refrigerant_r134a",       450, "kilogram",     "Tetrafluoroethane / R134a — cooling-system refrigerant top-up"),

    # --- Disposal ---
    ("diesel_building_machine",     1.0e6, "megajoule",     "Diesel, burned in building machine — demolition"),
    ("electricity_mv_serc",         1.4e4, "kilowatt hour", "Electricity MV, US-SERC — demolition (5.0e4 MJ / 3.6)"),
    ("tap_water",                   5.0e5, "kilogram",      "Tap water — demolition dust suppression"),
    ("transport_lorry_unspecified", 3.0e5, "ton kilometer",
     "Freight, lorry — debris to landfill (3,000 t x 100 km; source table's '3e8' looks like a typo for 3e5)"),

    # --- Recycling: scrap/waste sent to treatment ---
    ("scrap_aluminium",                     1.2e4, "kilogram", "Scrap aluminium to recycling"),
    ("scrap_copper",                        1.6e4, "kilogram", "Scrap copper to recycling"),
    ("scrap_steel",                         5.4e5, "kilogram", "Scrap steel to recycling"),
    ("weee_treatment",                      8.0e5, "kilogram", "Waste electric/electronic equipment (WEEE) treatment"),
    ("waste_polyurethane_treatment",        2.5e4, "kilogram", "Waste PUR insulation treatment"),
    ("waste_reinforced_concrete_treatment", 4.8e6, "kilogram", "Waste reinforced concrete treatment"),

    # --- Recycling: avoided-burden credits (see markdown note above re: cutoff vs apos) ---
    ("avoided_aluminium_hydroxide", -1.70e5, "kilogram", "Avoided aluminium hydroxide production"),
    ("avoided_metal_working",       -4.68e5, "kilogram", "Avoided metal working (credit derived from scrap steel treatment)"),
    ("avoided_steel",               -8.80e5, "kilogram", "Avoided virgin low-alloyed steel production"),
]
biosphere_exchanges_DC = []  # No direct foreground biosphere flows in the source study — all
                             # impact runs through the technosphere links above (background
                             # ecoinvent processes), matching the paper's own finding that
                             # >99% of impact is electricity-driven.
print("Data centre exchanges:", len(technosphere_exchanges_DC), "techno /",
      len(biosphere_exchanges_DC), "bio")


Data centre exchanges: 33 techno / 0 bio


## Write the foreground database

In [25]:
import bw2data as bd
FG_DB_NAME = FOREGROUND_DB
SMR_CODE   = "smr_hermesmann_1kg_h2"

if FG_DB_NAME in bd.databases:
    bd.Database(FG_DB_NAME).delete()

foreground_data = {
    (FG_DB_NAME, SMR_CODE): {
        "name": "Hydrogen production, SMR Hermesmann", "reference product": "hydrogen",
        "unit": "kilogram", "location": "GB", "database": FG_DB_NAME, "code": SMR_CODE,
        "comment": "Recreated from SimaPro export SMR.XLSX.",
        "exchanges": [{"input": (FG_DB_NAME, SMR_CODE), "amount": 1.0, "unit": "kilogram",
                        "type": "production", "name": "Hydrogen production, SMR Hermesmann"}],
    },
    (FG_DB_NAME, SMRCCS_CODE): {
        "name": "Hydrogen production, SMR-CCS Hermesmann", "reference product": "hydrogen",
        "unit": "kilogram", "location": "GB", "database": FG_DB_NAME, "code": SMRCCS_CODE,
        "comment": "Recreated from SimaPro export smr ccs.XLSX. CCS WT linked as foreground.",
        "exchanges": [
            {"input": (FG_DB_NAME, SMRCCS_CODE), "amount": 1.0, "unit": "kilogram",
             "type": "production", "name": "Hydrogen production, SMR-CCS Hermesmann"},
            {"input": (FG_DB_NAME, CCS_CODE), "amount": 8.01, "unit": "kilogram",
             "type": "technosphere", "name": "Carbon capture and storage, Hermesmann",
             "comment": "8.01 kg CO2 per kg H2 to foreground CCS waste-treatment"},
        ],
    },
    (FG_DB_NAME, CCS_CODE): {
        "name": "Carbon capture and storage, Hermesmann",
        "reference product": "carbon dioxide, captured",
        "unit": "kilogram", "location": "GB", "database": FG_DB_NAME, "code": CCS_CODE,
        "comment": "Foreground CCS waste treatment from ccs.XLSX. Treats 1 kg captured CO2.",
        "exchanges": [{"input": (FG_DB_NAME, CCS_CODE), "amount": 1.0, "unit": "kilogram",
                        "type": "production", "name": "Carbon capture and storage, Hermesmann"}],
    },
    (FG_DB_NAME, MP_E_CODE): {
        "name": "Hydrogen production, methane pyrolysis MP-E", "reference product": "hydrogen",
        "unit": "kilogram", "location": "DE", "database": FG_DB_NAME, "code": MP_E_CODE,
        "comment": "Methane pyrolysis MP-E from Table 3. Electric heating; no direct CO2.",
        "exchanges": [{"input": (FG_DB_NAME, MP_E_CODE), "amount": 1.0, "unit": "kilogram",
                        "type": "production", "name": "Hydrogen production, methane pyrolysis MP-E"}],
    },
    (FG_DB_NAME, AE_CODE): {
        "name": "Alkaline electrolyser, Hermesmann",
        "reference product": "alkaline electrolyser",
        "unit": "unit", "location": "GLO", "database": FG_DB_NAME, "code": AE_CODE,
        "comment": "AE manufacturing from alkaline electrolyser.XLSX. Capital good: 1 unit.",
        "exchanges": [{"input": (FG_DB_NAME, AE_CODE), "amount": 1.0, "unit": "unit",
                        "type": "production", "name": "Alkaline electrolyser, Hermesmann"}],
    },
    (FG_DB_NAME, AE_OP_CODE): {
        "name": "Hydrogen production, alkaline electrolysis Hermesmann",
        "reference product": "hydrogen", "unit": "kilogram", "location": "DE",
        "database": FG_DB_NAME, "code": AE_OP_CODE,
        "comment": "AE operation from alkaline operation.XLSX. AE capital good linked.",
        "exchanges": [
            {"input": (FG_DB_NAME, AE_OP_CODE), "amount": 1.0, "unit": "kilogram",
             "type": "production", "name": "Hydrogen production, alkaline electrolysis Hermesmann"},
            {"input": (FG_DB_NAME, AE_CODE), "amount": 3.24e-7, "unit": "unit",
             "type": "technosphere", "name": "Alkaline electrolyser, Hermesmann",
             "comment": "Capital good: 1 unit / 3,085,961 kg H2 lifetime = 3.24e-7 unit/kg H2"},
        ],
    },
    (FG_DB_NAME, PEM_CON_CODE): {
        "name": "PEM electrolyser, Hermesmann", "reference product": "PEM electrolyser",
        "unit": "unit", "location": "GB", "database": FG_DB_NAME, "code": PEM_CON_CODE,
        "comment": "PEM manufacturing from uk pem construction.XLSX. 1 unit (1 MW).",
        "exchanges": [{"input": (FG_DB_NAME, PEM_CON_CODE), "amount": 1.0, "unit": "unit",
                        "type": "production", "name": "PEM electrolyser, Hermesmann"}],
    },
    (FG_DB_NAME, PEM_OP_CODE): {
        "name": "Hydrogen production, PEM electrolysis Hermesmann",
        "reference product": "hydrogen", "unit": "kilogram", "location": "GB",
        "database": FG_DB_NAME, "code": PEM_OP_CODE,
        "comment": "PEM operation from ukpemoperation.XLSX. PEM capital good linked.",
        "exchanges": [
            {"input": (FG_DB_NAME, PEM_OP_CODE), "amount": 1.0, "unit": "kilogram",
             "type": "production", "name": "Hydrogen production, PEM electrolysis Hermesmann"},
            {"input": (FG_DB_NAME, PEM_CON_CODE), "amount": 3.37e-7, "unit": "unit",
             "type": "technosphere", "name": "PEM electrolyser, Hermesmann",
             "comment": "Capital good: 1 unit / 2,964,315 kg H2 lifetime = 3.37e-7 unit/kg H2"},
        ],
    },
    (FG_DB_NAME, SOEC_CON_CODE): {
        "name": "SOEC electrolyser, Hermesmann", "reference product": "SOEC electrolyser",
        "unit": "unit", "location": "GB", "database": FG_DB_NAME, "code": SOEC_CON_CODE,
        "comment": "SOEC manufacturing from uksoecconstruction.XLSX.",
        "exchanges": [{"input": (FG_DB_NAME, SOEC_CON_CODE), "amount": 1.0, "unit": "unit",
                        "type": "production", "name": "SOEC electrolyser, Hermesmann"}],
    },
    (FG_DB_NAME, SOEC_OP_CODE): {
        "name": "Hydrogen production, SOEC electrolysis Hermesmann",
        "reference product": "hydrogen", "unit": "kilogram", "location": "GB",
        "database": FG_DB_NAME, "code": SOEC_OP_CODE,
        "comment": "SOEC operation from uksoecoperation.XLSX. SOEC capital good linked.",
        "exchanges": [
            {"input": (FG_DB_NAME, SOEC_OP_CODE), "amount": 1.0, "unit": "kilogram",
             "type": "production", "name": "Hydrogen production, SOEC electrolysis Hermesmann"},
            {"input": (FG_DB_NAME, SOEC_CON_CODE), "amount": 2.645e-7, "unit": "unit",
             "type": "technosphere", "name": "SOEC electrolyser, Hermesmann",
             "comment": "Capital good: 1 unit / 3,779,894 kg H2 lifetime = 2.645e-7 unit/kg H2"},
        ],
    },
}

def _append_techno(target_code, exchanges, lookup):
    for key, amount, unit, comment in exchanges:
        act = lookup[key]
        foreground_data[(FG_DB_NAME, target_code)]["exchanges"].append({
            "input": act.key, "amount": amount, "unit": unit,
            "type": "technosphere", "name": act.get("name"), "comment": comment,
        })

def _append_bio(target_code, exchanges):
    for flow, amount, unit, comment in exchanges:
        foreground_data[(FG_DB_NAME, target_code)]["exchanges"].append({
            "input": flow.key, "amount": amount, "unit": unit,
            "type": "biosphere", "name": flow.get("name"), "comment": comment,
        })

_append_techno(SMR_CODE,      technosphere_exchanges,        activities)
_append_bio   (SMR_CODE,      biosphere_exchanges)
_append_techno(SMRCCS_CODE,   technosphere_exchanges_ccs,    activities_ccs)
_append_bio   (SMRCCS_CODE,   biosphere_exchanges_ccs)
_append_techno(CCS_CODE,      technosphere_exchanges_ccs_wt, activities_ccs)
_append_bio   (CCS_CODE,      biosphere_exchanges_ccs_wt)
_append_techno(MP_E_CODE,     technosphere_exchanges_MP_E,   activities_MP)
_append_techno(AE_CODE,       technosphere_exchanges_AE,     activities_AE)
_append_techno(AE_OP_CODE,    technosphere_exchanges_AE_op,  activities_AE_op)
_append_techno(PEM_CON_CODE,  technosphere_exchanges_PEM_con, activities_PEM_con)
_append_techno(PEM_OP_CODE,   technosphere_exchanges_PEM_op,  activities_PEM_op)
_append_techno(SOEC_CON_CODE, technosphere_exchanges_SOEC_con, activities_SOEC_con)
_append_techno(SOEC_OP_CODE,  technosphere_exchanges_SOEC_op,  activities_SOEC_op)

fg = bd.Database(FG_DB_NAME)
fg.write(foreground_data)
print("Foreground database written with 10 activities.\n")

# Refresh the shared handle exposed by lca_helpers.
fg_db = H.refresh_foreground()

# Hydrogen-producing activities (skip the construction-only and CCS waste-treatment codes).
H2_CODES = {
    "SMR":             SMR_CODE,
    "SMR-CCS":         SMRCCS_CODE,
    "MP-E":            MP_E_CODE,
    "AE operation":    AE_OP_CODE,
    "PEM operation":   PEM_OP_CODE,
    "SOEC operation":  SOEC_OP_CODE,
}

for label, code in H2_CODES.items():
    act = bd.get_activity((FG_DB_NAME, code))
    print("=" * 70)
    print(label, "→", act)
    for exc in act.exchanges():
        print(f"  {exc['type']:<14} {exc.amount:>12g}  "
              f"{exc.get('unit', ''):<15}  {exc.input.get('name', exc.input)}")

# =============================================================================
# Data centre facility (Zhang et al. 2025) — written to its own database so
# rebuilding the hydrogen foreground above never touches it, and vice versa.
# =============================================================================
DC_FG_DB_NAME = DC_FOREGROUND_DB

if DC_FG_DB_NAME in bd.databases:
    bd.Database(DC_FG_DB_NAME).delete()

foreground_data_dc = {
    (DC_FG_DB_NAME, DC_CODE): {
        "name": "Data centre facility, hyperscale, Virginia (Zhang et al. 2025)",
        "reference product": "data centre facility, 25-year cradle-to-grave",
        "unit": "unit", "location": "US-SERC", "database": DC_FG_DB_NAME, "code": DC_CODE,
        "comment": "Cradle-to-grave 25-year hyperscale data centre (10,000 m^2 white space), "
                   "Virginia US, SERC grid, PUE 1.4 baseline. Zhang et al. 2025, Cleaner Energy "
                   "Systems 12, 100223, Table 1 / supplementary Table S1. Functional unit is the "
                   "whole facility over its 25-year operating life, not a per-kWh or per-m^2 unit.",
        "exchanges": [{"input": (DC_FG_DB_NAME, DC_CODE), "amount": 1.0, "unit": "unit",
                        "type": "production",
                        "name": "Data centre facility, hyperscale, Virginia (Zhang et al. 2025)"}],
    },
}

for key, amount, unit, comment in technosphere_exchanges_DC:
    act = activities_DC[key]
    foreground_data_dc[(DC_FG_DB_NAME, DC_CODE)]["exchanges"].append({
        "input": act.key, "amount": amount, "unit": unit,
        "type": "technosphere", "name": act.get("name"), "comment": comment,
    })
for flow, amount, unit, comment in biosphere_exchanges_DC:
    foreground_data_dc[(DC_FG_DB_NAME, DC_CODE)]["exchanges"].append({
        "input": flow.key, "amount": amount, "unit": unit,
        "type": "biosphere", "name": flow.get("name"), "comment": comment,
    })

fg_dc = bd.Database(DC_FG_DB_NAME)
fg_dc.write(foreground_data_dc)
print(f"\nData centre foreground database written to {DC_FG_DB_NAME!r} with 1 activity.\n")

dc_act = bd.get_activity((DC_FG_DB_NAME, DC_CODE))
print("=" * 70)
print("Data centre facility", "→", dc_act)
for exc in dc_act.exchanges():
    print(f"  {exc['type']:<14} {exc.amount:>12g}  "
          f"{exc.get('unit', ''):<15}  {exc.input.get('name', exc.input)}")


/opt/miniconda3/envs/brightway/lib/python3.11/site-packages/bw2data/backends/base.py:899: UserWarning: 
            Please use `del databases['hydrogen foreground']` instead.
            Otherwise, the metadata and database get out of sync.
            Call `.delete(warn=False)` to skip this message in the future.
            
  warnings.warn(MESSAGE.format(self.name), UserWarning)
100%|██████████| 10/10 [00:00<00:00, 1181.36it/s]

08:46:51+0100 [info     ] Vacuuming database            


Foreground database written with 10 activities.

SMR → 'Hydrogen production, SMR Hermesmann' (kilogram, GB, None)
  production                1  kilogram         Hydrogen production, SMR Hermesmann
  technosphere           -1.1  kilowatt hour    market for electricity, high voltage
  technosphere           4.85  cubic meter      market for natural gas, high pressure
  technosphere           6.64  kilogram         market for tap water
  technosphere        6.6e-06  cubic meter      concrete, all types to generic market for concrete, normal strength
  technosphere        0.00506  kilogram         market for steel, unalloyed
  technosphere       4.17e-05  kilogram         market for aluminium, primary, cast alloy slab from continuous casting
  technosphere       6.18e-05  kilogram         market for cast iron
  technosphere        7.9e-10  unit             market for gas turbine, 10MW electrical
  technosphere        0.00172  cubic meter      market for wastewater, average
  biosphere    

/opt/miniconda3/envs/brightway/lib/python3.11/site-packages/bw2data/backends/base.py:899: UserWarning: 
            Please use `del databases['data centre foreground']` instead.
            Otherwise, the metadata and database get out of sync.
            Call `.delete(warn=False)` to skip this message in the future.
            
  warnings.warn(MESSAGE.format(self.name), UserWarning)
100%|██████████| 1/1 [00:00<00:00, 12157.40it/s]

08:46:57+0100 [info     ] Vacuuming database            



Data centre foreground database written to 'data centre foreground' with 1 activity.

Data centre facility → 'Data centre facility, hyperscale, Virginia (Zhang et al. 2025)' (unit, US-SERC, None)
  production                1  unit             Data centre facility, hyperscale, Virginia (Zhang et al. 2025)
  technosphere           4000  cubic meter      market for concrete, 30-32MPa
  technosphere         480000  kilogram         market for reinforcing steel
  technosphere         120000  kilogram         market for steel, low-alloyed, hot rolled
  technosphere          20000  kilogram         market for copper, cathode
  technosphere          15000  kilogram         market for aluminium alloy, metal matrix composite
  technosphere          25000  kilogram         market for polyurethane, rigid foam
  technosphere      3.036e+07  ton kilometer    market for transport, freight, sea, container ship
  technosphere         610000  ton kilometer    market for transport, freight, lorry 16-32